In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, Conv1D, MaxPooling1D, Flatten
from tensorflow.keras.callbacks import Callback

In [ ]:
df = pd.read_csv("/kaggle/input/cybersecurity-intrusion-detection-dataset/cybersecurity_intrusion_data.csv")

In [ ]:
df.drop("session_id", axis=1, inplace=True)

In [ ]:
categorical_cols = ['protocol_type', 'encryption_used', 'browser_type']
label_encoders = {}

In [ ]:
for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")  # Fill missing
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le
    print(f"Label mapping for {col}:", dict(zip(le.classes_, le.transform(le.classes_))))
    with open(f"{col}_label_encoder.pkl", "wb") as f:
        pickle.dump(le, f)

In [ ]:
X = df.drop("attack_detected", axis=1)
y = df["attack_detected"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

In [ ]:
X_train_cnn = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_test_cnn = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

In [ ]:
model = Sequential()
model.add(Input(shape=(X_train_cnn.shape[1], 1)))
model.add(Conv1D(filters=32, kernel_size=3, activation='relu'))
model.add(MaxPooling1D(pool_size=2))
model.add(Dropout(0.3))
model.add(Flatten())
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

class AccuracyCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        print(f"Epoch {epoch+1}: Accuracy = {logs['accuracy']:.4f}, Val Accuracy = {logs.get('val_accuracy', 0):.4f}")

In [ ]:
history = model.fit(
    X_train_cnn,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[AccuracyCallback()],
    verbose=0
)

In [ ]:
y_pred = model.predict(X_test_cnn)
y_pred_labels = (y_pred > 0.5).astype(int)

In [ ]:
print("\nClassification Report:\n", classification_report(y_test, y_pred_labels))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_labels))

model.save("cnn_trust_model.h5")

Label mapping for protocol_type: {'ICMP': 0, 'TCP': 1, 'UDP': 2}
Label mapping for encryption_used: {'AES': 0, 'DES': 1, 'Unknown': 2}
Label mapping for browser_type: {'Chrome': 0, 'Edge': 1, 'Firefox': 2, 'Safari': 3, 'Unknown': 4}
Epoch 1: Accuracy = 0.7213, Val Accuracy = 0.7975
Epoch 2: Accuracy = 0.7949, Val Accuracy = 0.8349
Epoch 3: Accuracy = 0.8150, Val Accuracy = 0.8460
Epoch 4: Accuracy = 0.8327, Val Accuracy = 0.8512
Epoch 5: Accuracy = 0.8347, Val Accuracy = 0.8571
Epoch 6: Accuracy = 0.8420, Val Accuracy = 0.8617
Epoch 7: Accuracy = 0.8442, Val Accuracy = 0.8630
Epoch 8: Accuracy = 0.8491, Val Accuracy = 0.8650
Epoch 9: Accuracy = 0.8534, Val Accuracy = 0.8657
Epoch 10: Accuracy = 0.8520, Val Accuracy = 0.8676
Epoch 11: Accuracy = 0.8519, Val Accuracy = 0.8611
Epoch 12: Accuracy = 0.8581, Val Accuracy = 0.8755
Epoch 13: Accuracy = 0.8565, Val Accuracy = 0.8761
Epoch 14: Accuracy = 0.8570, Val Accuracy = 0.8781
Epoch 15: Accuracy = 0.8611, Val Accuracy = 0.8755
Epoch 16: A